In [15]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib

In [16]:
df = pd.read_csv("ventore_sales_10_menus.csv")
df["date"] = pd.to_datetime(df["date"], dayfirst=False, errors="coerce")

df = df.sort_values(["date", "menu_id"]).reset_index(drop=True)

df["day"] = df["date"].dt.day
df["year"] = df["date"].dt.year

df["qty_sold_lag_1"] = df.groupby("menu_id")["qty_sold"].shift(1)
df["qty_sold_lag_7"] = df.groupby("menu_id")["qty_sold"].shift(7)

df = df.dropna(subset=["qty_sold_lag_7"]).reset_index(drop=True)

drop_cols = ["qty_sold", "gross_sales", "date", "menu_name", "day_name"]
features = df.drop(columns=drop_cols)
target = df["qty_sold"]

numeric_features = [
    "day_of_week",
    "is_weekend",
    "month",
    "week_of_year",
    "unit_price",
    "day",
    "year",
    "event_flag",
    "qty_sold_lag_1", 
    "qty_sold_lag_7", 
]

categorical_features = [
    "weather",
    "event_name",
    "menu_id",
]

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42,
    shuffle=False,
)

In [ ]:
numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        XGBRegressor(
            n_estimators=200,
            learning_rate=0.05, 
            max_depth=6,
            random_state=42,
            objective="reg:squarederror",
            tree_method="hist", 
        ),
    ),
])

In [18]:
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test RMSE: {rmse:.2f}")
print(f"Test R2: {r2:.4f}")

joblib.dump(pipeline, "xgb_qty_sold_pipeline.joblib")
print("Model saved to xgb_qty_sold_pipeline.joblib")

Test RMSE: 5.23
Test R2: 0.9774
Model saved to xgb_qty_sold_pipeline.joblib
